In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')

pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from advanced_evaluation.advanced_evaluation import SampleOutcomesAdvanced

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
#model_name = 'bpic_2017_all_2'
model_name = 'helpdesk'

n_processes = 32

log_name = 'test'

with open('../transformed_event_logs/Helpdesk_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)


test_event_log['Case ID'] = test_event_log['Case ID'].astype(str)
test_event_log['case:concept:name'] = test_event_log['Case ID']
test_event_log['time:timestamp_start'] = test_event_log['Complete Timestamp_start']
test_event_log['time:timestamp_complete'] = test_event_log['Complete Timestamp_complete']


known_resources = ['Value 1', 'Value 10', 'Value 11', 'Value 12', 'Value 13', 'Value 14', 'Value 15', 'Value 16', 'Value 17', 'Value 18', 'Value 19', 'Value 2', 'Value 20', 'Value 21', 'Value 22', 'Value 3', 'Value 4', 'Value 5', 'Value 6', 'Value 7', 'Value 8', 'Value 9']
known_activities = ['Assign seriousness', 'Closed', 'Create SW anomaly', 'DUPLICATE', 'INVALID', 'Insert ticket', 'RESOLVED', 'Require upgrade', 'Resolve SW anomaly', 'Resolve ticket', 'Schedule intervention', 'Take in charge ticket', 'VERIFIED', 'Wait']
ii1 = ['intercase_n_1__Assign seriousness', 'intercase_n_1__Closed', 'intercase_n_1__Create SW anomaly', 'intercase_n_1__DUPLICATE', 'intercase_n_1__INVALID', 'intercase_n_1__Insert ticket', 'intercase_n_1__RESOLVED', 'intercase_n_1__Require upgrade', 'intercase_n_1__Resolve SW anomaly', 'intercase_n_1__Resolve ticket', 'intercase_n_1__Schedule intervention', 'intercase_n_1__Take in charge ticket', 'intercase_n_1__VERIFIED', 'intercase_n_1__Wait']
ii3 = ['intercase_n_3__Assign seriousness', 'intercase_n_3__Assign seriousness_Assign seriousness', 'intercase_n_3__Assign seriousness_Assign seriousness_Assign seriousness', 'intercase_n_3__Assign seriousness_Assign seriousness_Resolve ticket', 'intercase_n_3__Assign seriousness_Assign seriousness_Take in charge ticket', 'intercase_n_3__Assign seriousness_Assign seriousness_Wait', 'intercase_n_3__Assign seriousness_Create SW anomaly', 'intercase_n_3__Assign seriousness_Create SW anomaly_Create SW anomaly', 'intercase_n_3__Assign seriousness_Create SW anomaly_Require upgrade', 'intercase_n_3__Assign seriousness_Create SW anomaly_Take in charge ticket', 'intercase_n_3__Assign seriousness_Require upgrade', 'intercase_n_3__Assign seriousness_Require upgrade_Require upgrade', 'intercase_n_3__Assign seriousness_Require upgrade_Resolve ticket', 'intercase_n_3__Assign seriousness_Resolve ticket', 'intercase_n_3__Assign seriousness_Resolve ticket_Closed', 'intercase_n_3__Assign seriousness_Resolve ticket_Resolve ticket', 'intercase_n_3__Assign seriousness_Resolve ticket_Take in charge ticket', 'intercase_n_3__Assign seriousness_Resolve ticket_Wait', 'intercase_n_3__Assign seriousness_Take in charge ticket', 'intercase_n_3__Assign seriousness_Take in charge ticket_Assign seriousness', 'intercase_n_3__Assign seriousness_Take in charge ticket_Create SW anomaly', 'intercase_n_3__Assign seriousness_Take in charge ticket_Require upgrade', 'intercase_n_3__Assign seriousness_Take in charge ticket_Resolve SW anomaly', 'intercase_n_3__Assign seriousness_Take in charge ticket_Resolve ticket', 'intercase_n_3__Assign seriousness_Take in charge ticket_Schedule intervention', 'intercase_n_3__Assign seriousness_Take in charge ticket_Take in charge ticket', 'intercase_n_3__Assign seriousness_Take in charge ticket_Wait', 'intercase_n_3__Assign seriousness_Wait', 'intercase_n_3__Assign seriousness_Wait_Assign seriousness', 'intercase_n_3__Assign seriousness_Wait_Resolve ticket', 'intercase_n_3__Assign seriousness_Wait_Take in charge ticket', 'intercase_n_3__Assign seriousness_Wait_Wait', 'intercase_n_3__Closed_Take in charge ticket_Resolve ticket', 'intercase_n_3__Create SW anomaly', 'intercase_n_3__Create SW anomaly_Create SW anomaly_Resolve SW anomaly', 'intercase_n_3__Create SW anomaly_Create SW anomaly_Resolve ticket', 'intercase_n_3__Create SW anomaly_Require upgrade_Require upgrade', 'intercase_n_3__Create SW anomaly_Require upgrade_Resolve ticket', 'intercase_n_3__Create SW anomaly_Require upgrade_VERIFIED', 'intercase_n_3__Create SW anomaly_Resolve SW anomaly', 'intercase_n_3__Create SW anomaly_Resolve SW anomaly_Resolve SW anomaly', 'intercase_n_3__Create SW anomaly_Resolve SW anomaly_Resolve ticket', 'intercase_n_3__Create SW anomaly_Resolve ticket_RESOLVED', 'intercase_n_3__Create SW anomaly_Resolve ticket_Take in charge ticket', 'intercase_n_3__Create SW anomaly_Take in charge ticket_Create SW anomaly', 'intercase_n_3__Create SW anomaly_Take in charge ticket_Wait', 'intercase_n_3__Insert ticket', 'intercase_n_3__Insert ticket_Assign seriousness', 'intercase_n_3__Insert ticket_Assign seriousness_Assign seriousness', 'intercase_n_3__Insert ticket_Assign seriousness_Resolve ticket', 'intercase_n_3__Insert ticket_Assign seriousness_Take in charge ticket', 'intercase_n_3__Insert ticket_Take in charge ticket', 'intercase_n_3__Insert ticket_Take in charge ticket_Resolve ticket', 'intercase_n_3__Insert ticket_Wait', 'intercase_n_3__Insert ticket_Wait_Wait', 'intercase_n_3__RESOLVED_INVALID_Closed', 'intercase_n_3__RESOLVED_INVALID_VERIFIED', 'intercase_n_3__Require upgrade_Create SW anomaly_Resolve ticket', 'intercase_n_3__Require upgrade_Require upgrade_Create SW anomaly', 'intercase_n_3__Require upgrade_Require upgrade_Require upgrade', 'intercase_n_3__Require upgrade_Require upgrade_Resolve ticket', 'intercase_n_3__Require upgrade_Require upgrade_Take in charge ticket', 'intercase_n_3__Require upgrade_Resolve ticket_Resolve ticket', 'intercase_n_3__Require upgrade_Take in charge ticket_Resolve ticket', 'intercase_n_3__Require upgrade_Take in charge ticket_Wait', 'intercase_n_3__Require upgrade_VERIFIED_DUPLICATE', 'intercase_n_3__Require upgrade_Wait_Resolve ticket', 'intercase_n_3__Resolve SW anomaly_Require upgrade_Create SW anomaly', 'intercase_n_3__Resolve SW anomaly_Require upgrade_Resolve ticket', 'intercase_n_3__Resolve SW anomaly_Resolve SW anomaly_Require upgrade', 'intercase_n_3__Resolve SW anomaly_Resolve SW anomaly_Resolve ticket', 'intercase_n_3__Resolve ticket', 'intercase_n_3__Resolve ticket_Assign seriousness_Take in charge ticket', 'intercase_n_3__Resolve ticket_Closed_Take in charge ticket', 'intercase_n_3__Resolve ticket_RESOLVED_INVALID', 'intercase_n_3__Resolve ticket_Require upgrade_Take in charge ticket', 'intercase_n_3__Resolve ticket_Resolve ticket_Require upgrade', 'intercase_n_3__Resolve ticket_Resolve ticket_Resolve ticket', 'intercase_n_3__Resolve ticket_Resolve ticket_Take in charge ticket', 'intercase_n_3__Resolve ticket_Take in charge ticket', 'intercase_n_3__Resolve ticket_Take in charge ticket_Create SW anomaly', 'intercase_n_3__Resolve ticket_Take in charge ticket_Require upgrade', 'intercase_n_3__Resolve ticket_Take in charge ticket_Resolve ticket', 'intercase_n_3__Resolve ticket_Take in charge ticket_Take in charge ticket', 'intercase_n_3__Resolve ticket_Take in charge ticket_Wait', 'intercase_n_3__Resolve ticket_Wait_Resolve ticket', 'intercase_n_3__Resolve ticket_Wait_Wait', 'intercase_n_3__Schedule intervention_Take in charge ticket_Wait', 'intercase_n_3__Take in charge ticket', 'intercase_n_3__Take in charge ticket_Assign seriousness_Assign seriousness', 'intercase_n_3__Take in charge ticket_Create SW anomaly_Create SW anomaly', 'intercase_n_3__Take in charge ticket_Create SW anomaly_Require upgrade', 'intercase_n_3__Take in charge ticket_Create SW anomaly_Resolve SW anomaly', 'intercase_n_3__Take in charge ticket_Create SW anomaly_Resolve ticket', 'intercase_n_3__Take in charge ticket_Create SW anomaly_Take in charge ticket', 'intercase_n_3__Take in charge ticket_Require upgrade_Create SW anomaly', 'intercase_n_3__Take in charge ticket_Require upgrade_Require upgrade', 'intercase_n_3__Take in charge ticket_Require upgrade_Resolve ticket', 'intercase_n_3__Take in charge ticket_Require upgrade_Take in charge ticket', 'intercase_n_3__Take in charge ticket_Require upgrade_Wait', 'intercase_n_3__Take in charge ticket_Resolve SW anomaly_Resolve ticket', 'intercase_n_3__Take in charge ticket_Resolve ticket', 'intercase_n_3__Take in charge ticket_Resolve ticket_Assign seriousness', 'intercase_n_3__Take in charge ticket_Resolve ticket_Closed', 'intercase_n_3__Take in charge ticket_Resolve ticket_Resolve ticket', 'intercase_n_3__Take in charge ticket_Resolve ticket_Take in charge ticket', 'intercase_n_3__Take in charge ticket_Resolve ticket_Wait', 'intercase_n_3__Take in charge ticket_Schedule intervention_Resolve ticket', 'intercase_n_3__Take in charge ticket_Schedule intervention_Take in charge ticket', 'intercase_n_3__Take in charge ticket_Take in charge ticket', 'intercase_n_3__Take in charge ticket_Take in charge ticket_Create SW anomaly', 'intercase_n_3__Take in charge ticket_Take in charge ticket_Require upgrade', 'intercase_n_3__Take in charge ticket_Take in charge ticket_Resolve ticket', 'intercase_n_3__Take in charge ticket_Take in charge ticket_Schedule intervention', 'intercase_n_3__Take in charge ticket_Take in charge ticket_Take in charge ticket', 'intercase_n_3__Take in charge ticket_Take in charge ticket_Wait', 'intercase_n_3__Take in charge ticket_Wait', 'intercase_n_3__Take in charge ticket_Wait_Create SW anomaly', 'intercase_n_3__Take in charge ticket_Wait_Require upgrade', 'intercase_n_3__Take in charge ticket_Wait_Resolve ticket', 'intercase_n_3__Take in charge ticket_Wait_Take in charge ticket', 'intercase_n_3__Take in charge ticket_Wait_Wait', 'intercase_n_3__VERIFIED_DUPLICATE_Resolve ticket', 'intercase_n_3__Wait', 'intercase_n_3__Wait_Assign seriousness_Assign seriousness', 'intercase_n_3__Wait_Assign seriousness_Take in charge ticket', 'intercase_n_3__Wait_Create SW anomaly_Require upgrade', 'intercase_n_3__Wait_Create SW anomaly_Resolve ticket', 'intercase_n_3__Wait_Require upgrade_Require upgrade', 'intercase_n_3__Wait_Require upgrade_Resolve ticket', 'intercase_n_3__Wait_Require upgrade_Wait', 'intercase_n_3__Wait_Resolve ticket', 'intercase_n_3__Wait_Resolve ticket_Resolve ticket', 'intercase_n_3__Wait_Resolve ticket_Take in charge ticket', 'intercase_n_3__Wait_Resolve ticket_Wait', 'intercase_n_3__Wait_Take in charge ticket_Create SW anomaly', 'intercase_n_3__Wait_Take in charge ticket_Require upgrade', 'intercase_n_3__Wait_Take in charge ticket_Resolve ticket', 'intercase_n_3__Wait_Take in charge ticket_Take in charge ticket', 'intercase_n_3__Wait_Take in charge ticket_Wait', 'intercase_n_3__Wait_Wait_Create SW anomaly', 'intercase_n_3__Wait_Wait_Resolve ticket', 'intercase_n_3__Wait_Wait_Take in charge ticket', 'intercase_n_3__Wait_Wait_Wait']

In [3]:
drbart_model_path = '../../../models/advanced/'+model_name+'/crsdacrc_ii1/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'resources' : True,
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)',
                                                                              '(lambda inter_instance_counts, inter_instance_column_names : [0 if inter_instance_column_name not in inter_instance_counts else inter_instance_counts[inter_instance_column_name] for inter_instance_column_name in inter_instance_column_names])(inter_instance_counts, self.inter_instance_column_names)'
                                                                             ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'inter_instance_column_names' : ii1,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 904/904 [00:10<00:00, 83.93it/s] 


In [4]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-5.049283156887369043400387819')

In [5]:
np.mean(get_pscores(likelihoods_A))

np.float64(1428563.2331136449)

In [6]:
drbart_model_path = '../../../models/advanced/'+model_name+'/crsdacrc_ii3/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'resources' : True,
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)',
                                                                              '(lambda inter_instance_counts, inter_instance_column_names : [0 if inter_instance_column_name not in inter_instance_counts else inter_instance_counts[inter_instance_column_name] for inter_instance_column_name in inter_instance_column_names])(inter_instance_counts, self.inter_instance_column_names)'
                                                                             ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'inter_instance_column_names' : ii3,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 904/904 [00:10<00:00, 84.42it/s] 


In [7]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-5.287698689304632464858880781')

In [8]:
np.mean(get_pscores(likelihoods_A))

np.float64(1584918.2830701154)